## Случайный лес без масштабирования

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures
from sklearn.ensemble import RandomForestClassifier
from joblib import parallel_backend
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate

# Загрузка данных
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
X = train.drop(columns=["Id", "Survived"]).copy()
y = train["Survived"]
X_test = test.drop(columns="Id").copy()
for data in [X, X_test]:
    data["around_whealth"] = (data["Parch"] + data["SibSp"]) * data["Fare"]


## Подготовка признаков

In [2]:
# Преобразования внутри фолдов
numeric_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare", "around_whealth"]
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Пропуск")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_columns),
    ("categorical", categorical_transformer, ["Sex", "Embarked"]),
])
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=1)),
])


## Подбор параметров

In [3]:
# Пятикратная кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {"model__max_depth": [5, None], "model__min_samples_leaf": [1, 4], "model__max_features": ["sqrt", 0.8]}
search = GridSearchCV(
    pipeline, param_grid, scoring={"f1": "f1", "roc_auc": "roc_auc"},
    refit="f1", cv=cv, n_jobs=4, return_train_score=True,
)
with parallel_backend("threading"):
    search.fit(X, y)
print("Лучшие параметры:", search.best_params_)
print("F1 при подборе:", round(search.best_score_, 4))
cv_results = pd.DataFrame(search.cv_results_)
display(cv_results[["params", "mean_train_f1", "mean_test_f1", "std_test_f1", "mean_test_roc_auc"]]
        .sort_values("mean_test_f1", ascending=False).head(10))


Лучшие параметры: {'model__max_depth': None, 'model__max_features': 0.8, 'model__min_samples_leaf': 4}
F1 при подборе: 0.7647


,params,mean_train_f1,mean_test_f1,std_test_f1,mean_test_roc_auc
7,"{'model__max_depth': None, 'model__max_feature...",0.872946,0.764664,0.037624,0.864759
5,"{'model__max_depth': None, 'model__max_feature...",0.865916,0.757544,0.049931,0.872617
2,"{'model__max_depth': 5, 'model__max_features':...",0.837782,0.755147,0.048204,0.867257
3,"{'model__max_depth': 5, 'model__max_features':...",0.823558,0.753368,0.055266,0.864407
0,"{'model__max_depth': 5, 'model__max_features':...",0.817636,0.743733,0.044809,0.874713
1,"{'model__max_depth': 5, 'model__max_features':...",0.811087,0.740280,0.039833,0.870027
4,"{'model__max_depth': None, 'model__max_feature...",0.976338,0.739409,0.034719,0.857377
6,"{'model__max_depth': None, 'model__max_feature...",0.976338,0.739157,0.035123,0.855181


## Независимая оценка подбора

In [4]:
# Вложенная кросс-валидация
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
with parallel_backend("threading"):
    nested_scores = cross_validate(
        search, X, y, cv=outer_cv, scoring={"f1": "f1", "roc_auc": "roc_auc"},
        n_jobs=1,
    )
fold_results = pd.DataFrame({
    "Фолд": range(1, 6),
    "F1": nested_scores["test_f1"],
    "ROC AUC": nested_scores["test_roc_auc"],
})
display(fold_results)
print("F1:", round(fold_results["F1"].mean(), 4), "±", round(fold_results["F1"].std(ddof=0), 4))
print("ROC AUC:", round(fold_results["ROC AUC"].mean(), 4), "±", round(fold_results["ROC AUC"].std(ddof=0), 4))


,Фолд,F1,ROC AUC
0,1,0.753846,0.862187
1,2,0.803150,0.869118
2,3,0.778626,0.861230
3,4,0.739130,0.854746
4,5,0.724409,0.865975


F1: 0.7598 ± 0.0281
ROC AUC: 0.8627 ± 0.0048


## Предсказания теста

In [5]:
# Сохранение посылки
best_model = search.best_estimator_
submission = pd.DataFrame({"Id": test["Id"], "Survived": best_model.predict(X_test)})
sample = pd.read_csv("data/sample_submission.csv")
assert submission.columns.tolist() == sample.columns.tolist()
assert submission["Id"].equals(sample["Id"])
assert submission["Survived"].isin([0, 1]).all()
assert not submission.isna().any().any()
submission.to_csv("submission_random_forest_unscaled_cv.csv", index=False)
print("Сохранён submission_random_forest_unscaled_cv.csv")


Сохранён submission_random_forest_unscaled_cv.csv
